# To Bugfix

prepare.py        | steps.py                  | pipeline/                | status
--------------------------------------------------------------------------
prepare_dataset() | download()                | build_manifest()         | X
                  |                           | download_methylation()   | X
                  |                           | initialize_audit_table() | X
                  |                           | build_metadata()         | X
                  |                           | build_biospecimen()      | X
                  |                           | update_metadata()        | X
                  |                           | update_biospecimen()     | X
                  | clean_data()              |                          | X
                  | load_raw_project()        |                          | X
                  | quality_control()         | load_annotation()        | X
                  |                           | sample_qc()              | X
                  |                           | probe_qc()               | X
                  |                           | clean_metadata()         | X
                  | preprocess()              | normalize()              | X
                  |                           | filter_variance()        | X
                  |                           | convert_to_mval()        | X
                  |                           | impute()                 | X
                  | save_project()            |                          | X
prepare_cohort()  | load_processed_project()  |                          |
                  | aggregate_cohort()        | cohort_aggregation()     |
                  | cohort_batch_correction() | batch_correct()          |
                  | aggregate_genes()         | gene_aggregation()       |
                  | winsorize()               |                          |
                  | split()                   |                          |

%load_ext autoreload
%autoreload 2


## Initial Set-up

In [1]:
from methyltrain.config.loader import load_config
from methyltrain.fs.layout import ProjectLayout, CohortLayout

layout = ProjectLayout(
    project_name = "TCGA-CHOL",
    root_dir = "../data",
    raw_dir = "../data/raw/TCGA-CHOL",
    audit_table = "../data/metadata/TCGA-CHOL/TCGA-CHOL_audit_table.csv",
    metadata = "../data/metadata/TCGA-CHOL/TCGA-CHOL_metadata.csv",
    manifest = "../data/metadata/TCGA-CHOL/TCGA-CHOL_manifest.csv",
    status_log = "../data/metadata/TCGA-CHOL/TCGA-CHOL_status_log.csv",
    project_adata = "../data/processed/TCGA-CHOL_adata.h5ad"
)
layout.initialize()
layout.validate()

config = load_config(
    "/Volumes/FBI_Drive/MethylTrain/config/TCGA-CHOL_config_local.yaml"
)

## Current Debug

In [2]:
from typing import Dict, Tuple

import anndata as ad
import pandas as pd

from methyltrain.fs.layout import ProjectLayout, CohortLayout
from methyltrain.api.steps import (
    download, 
    clean_data, 
    quality_control, 
    preprocess, 
    aggregate_cohort, 
    cohort_batch_correction,
    aggregate_genes,
    winsorize,
    split,
    load_raw_project,
    load_processed_project
)

from methyltrain.pipeline.preprocess import (
    normalize, filter_variance, impute, convert_to_mval
)
from methyltrain.utils.load_utils import load_audit_table